# 🚛 Análisis Financiero TCO a 5 Años: Diésel vs Eléctrico (44t)

Este notebook comprueba la viabilidad financiera de las inversiones en flota pesada para diferentes escenarios (Compra, Leasing y Renting) a lo largo de un horizonte operativo de 5 años.

## 📐 Modelo Matemático del TCO

El análisis se basa en el **Valor Actual Neto (VAN)** de todos los flujos de caja asociados a la vida útil del activo. Un VAN más negativo representa un coste total real mayor.

### 1. Valor Actual Neto (VAN)
La fórmula principal para descontar los flujos de caja futuros al presente es:

$$VAN_{TCO} = \sum_{t=0}^{n} \frac{CF_t^{net}}{(1+WACC)^t}$$

Donde:
- $CF_t^{net}$: Flujo de caja neto en el año $t$.
- $WACC$: Coste promedio ponderado del capital (Tasa de descuento).
- $n$: Horizonte temporal (5 años).

### 2. Flujo de Caja Neto ($CF_t^{net}$)
Para cada año, el flujo neto se compone de la inversión inicial, los costes operativos, el ahorro fiscal y el valor de recuperación:

$$CF_t^{net} = -OPEX_t - Cuota_t + TS_t + RV_t + S_t$$

Siendo:
- $OPEX_t$: Gastos operativos (Energía, Mantenimiento, Seguros).
- $Cuota_t$: Pago por financiación (Leasing) o alquiler (Renting).
- $TS_t$ (**Tax Shield**): Escudo fiscal derivado de gastos deducibles.
- $RV_t$: Valor residual recuperado al final del periodo.
- $S_t$: Subvenciones gubernamentales (e.g., MOVES III).

### 3. Escudo Fiscal (Tax Shield)
El ahorro en impuestos de sociedades ($\tau = 25\%$) se calcula sobre los gastos deducibles:

$$TS_t = (OPEX_t + Depr_t + Intereses_t) \times \tau$$

Donde $Depr_t$ es la amortización contable anual del activo.

In [4]:
import sys
import os
import pandas as pd

# Asegurar que el módulo logistic_core está en el path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from logistic_core.utils.investment_analyzer import InvestmentAnalyzer


WARNING  GOOGLE_MAPS_API_KEY no está configurada. Se usará estimación Haversine.

### 1. Parámetros de Simulación
Definimos el uso intensivo anual del camión, inflación constante y coste de capital.

In [5]:
# Escenario: Transporte Intensivo para 44t
kms_anuales = 130_000
wacc = 0.07
inflación = 0.02

analyzer = InvestmentAnalyzer(
    kms_anuales=kms_anuales,
    wacc=wacc,
    inflación_anual=inflación
)

print(f"Distancia Total a 5 Años: {kms_anuales * 5:,.0f} km".replace(',', '.'))

Distancia Total a 5 Años: 650.000 km


### 2. Generación del Reporte Comparativo
Iteramos sobre tecnologías y modos de financiación para crear un consolidado de resultados netos (VAN Acumulado).

In [6]:
data_results = []

for tec in ["diesel", "electrico"]:
    res_compra = analyzer.evaluar_compra(tec)
    res_leasing = analyzer.evaluar_leasing(tec)
    res_renting = analyzer.evaluar_renting(tec)
    
    tecnologia_str = "Diésel" if tec == "diesel" else "Eléctrico"
    
    data_results.append({
        "Tecnología": tecnologia_str,
        "Modalidad": "Compra Propiedad",
        "TCO Neto (VAN) €": -res_compra["tco_van_acumulado"],
        "Coste por KM €/km": res_compra["coste_neto_por_km"]
    })
    
    data_results.append({
        "Tecnología": tecnologia_str,
        "Modalidad": "Leasing Financiero",
        "TCO Neto (VAN) €": -res_leasing["tco_van_acumulado"],
        "Coste por KM €/km": res_leasing["coste_neto_por_km"]
    })
    
    data_results.append({
        "Tecnología": tecnologia_str,
        "Modalidad": "Renting Operativo",
        "TCO Neto (VAN) €": -res_renting["tco_van_acumulado"],
        "Coste por KM €/km": res_renting["coste_neto_por_km"]
    })

df_results = pd.DataFrame(data_results)

# Formateo visual para lectura fácil
df_results.style.format({
    "TCO Neto (VAN) €": "{:,.0f} €",
    "Coste por KM €/km": "{:.3f} €"
}).background_gradient(subset=["TCO Neto (VAN) €"], cmap="RdYlGn_r")

,Tecnología,Modalidad,TCO Neto (VAN) €,Coste por KM €/km
0,Diésel,Compra Propiedad,"335,248 €",0.516 €
1,Diésel,Leasing Financiero,"342,353 €",0.527 €
2,Diésel,Renting Operativo,"309,034 €",0.475 €
3,Eléctrico,Compra Propiedad,"351,564 €",0.541 €
4,Eléctrico,Leasing Financiero,"318,610 €",0.490 €
5,Eléctrico,Renting Operativo,"305,695 €",0.470 €


### 3. Conclusión del TCO
El VAN acumulado indica la salida real de caja. Cuanto **menor sea el VAN (verde)**, financieramente mejor ha sido la estrategia frente a los kilómetros recorridos.